# Capítulo 12 · Generadores

*Traducción y adaptación de [A Whirlwind Tour of Python](https://github.com/jakevdp/WhirlwindTourOfPython)
de Jake VanderPlas (O'Reilly, 2016), publicado con licencia [CC0](https://creativecommons.org/publicdomain/zero/1.0/deed.es).
Adaptación: [MiAtlas.cl](https://miatlas.cl) · Licencia [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es).*

📖 Lectura complementaria del curso de Python de MiAtlas.cl · se abre en Colab o en JupyterLite, sin GPU.

Aquí profundizaremos en los generadores de Python, tanto en las *expresiones generadoras* como en las *funciones generadoras*.

## Expresiones generadoras

La diferencia entre las comprensiones de listas y las expresiones generadoras a veces confunde; veamos rápidamente en qué se distinguen:

### Las comprensiones de listas usan corchetes; las expresiones generadoras, paréntesis
Esta es una comprensión de listas típica:

In [ ]:
[n ** 2 for n in range(12)]

Y esta es una expresión generadora típica:

In [ ]:
(n ** 2 for n in range(12))

Fíjate en que mostrar la expresión generadora no muestra su contenido. Una forma de ver el contenido de una expresión generadora es pasarla al constructor ``list``:

In [ ]:
G = (n ** 2 for n in range(12))
list(G)

### Una lista es una colección de valores; un generador es una receta para producir valores
Cuando creas una lista, construyes de verdad una colección de valores, y eso tiene un costo en memoria.
Cuando creas un generador, no construyes una colección de valores, sino una receta para producirlos.
Ambos ofrecen la misma interfaz de iterador, como se ve aquí:

In [ ]:
L = [n ** 2 for n in range(12)]
for val in L:
    print(val, end=' ')

In [ ]:
G = (n ** 2 for n in range(12))
for val in G:
    print(val, end=' ')

La diferencia es que una expresión generadora no calcula los valores hasta que se necesitan.
Eso no solo ahorra memoria, ¡también ahorra cálculo!
Además, significa que el tamaño de una lista está limitado por la memoria disponible, mientras que el de una expresión generadora es ilimitado.

> 📝 **Nota de MiAtlas.cl:** puedes medir la diferencia de memoria con ``sys.getsizeof``. La lista ocupa más mientras más elementos tiene; el generador ocupa siempre lo mismo, porque solo guarda la receta y el punto en que va. (Los números exactos dependen de la versión de Python y del equipo.)

In [ ]:
import sys

lista = [n ** 2 for n in range(100_000)]
generador = (n ** 2 for n in range(100_000))
print(f"Lista:     {sys.getsizeof(lista):>9,} bytes")
print(f"Generador: {sys.getsizeof(generador):>9,} bytes")

Un ejemplo de expresión generadora infinita se puede crear con el iterador ``count`` del módulo ``itertools``:

In [ ]:
from itertools import count
count()

In [ ]:
for i in count():
    print(i, end=' ')
    if i >= 10: break

El iterador ``count`` seguirá contando felizmente para siempre hasta que le digas que pare; eso lo hace práctico para crear generadores que también sigan para siempre:

In [ ]:
factores = [2, 3, 5, 7]
G = (i for i in count() if all(i % n > 0 for n in factores))
for val in G:
    print(val, end=' ')
    if val > 40: break

Quizás ya ves hacia dónde vamos: si ampliáramos la lista de factores de la manera adecuada, tendríamos el comienzo de un generador de números primos con el algoritmo de la criba de Eratóstenes. Lo exploraremos en un momento.

> 📝 **Nota de MiAtlas.cl:** para tomar solo los primeros valores de un generador infinito, sin escribir el ``break`` a mano, existe ``itertools.islice``. Además, cuando una expresión generadora es el único argumento de una función, puedes omitir los paréntesis extra: ``sum(n ** 2 for n in range(12))``.

In [ ]:
from itertools import islice

print(list(islice(count(), 11)))       # los 11 primeros valores de count()
print(sum(n ** 2 for n in range(12)))  # suma sin construir ninguna lista

### Una lista se puede recorrer muchas veces; una expresión generadora, solo una
Esta es una de las posibles trampas de las expresiones generadoras.
Con una lista podemos hacer esto sin problema:

In [ ]:
L = [n ** 2 for n in range(12)]
for val in L:
    print(val, end=' ')
print()

for val in L:
    print(val, end=' ')

Una expresión generadora, en cambio, se agota después de un recorrido:

In [ ]:
G = (n ** 2 for n in range(12))
list(G)

In [ ]:
list(G)

Esto puede ser muy útil, porque significa que el recorrido se puede detener y retomar:

In [ ]:
G = (n**2 for n in range(12))
for n in G:
    print(n, end=' ')
    if n > 30: break

print("\nhaciendo otra cosa entremedio")

for n in G:
    print(n, end=' ')

Un caso en que esto me ha resultado útil es al trabajar con colecciones de archivos de datos en disco: permite analizarlos por lotes con mucha facilidad, y el generador lleva la cuenta de cuáles quedan por revisar.

## Funciones generadoras: usar ``yield``
En la sección anterior vimos que las comprensiones de listas sirven mejor para crear listas relativamente simples, y que en situaciones más complicadas puede convenir un bucle ``for`` normal.
Con las expresiones generadoras pasa lo mismo: podemos crear generadores más complejos con *funciones generadoras*, que usan la instrucción ``yield``.

Aquí tenemos dos formas de construir la misma lista:

In [ ]:
L1 = [n ** 2 for n in range(12)]

L2 = []
for n in range(12):
    L2.append(n ** 2)

print(L1)
print(L2)

Del mismo modo, aquí tenemos dos formas de construir generadores equivalentes:

In [ ]:
G1 = (n ** 2 for n in range(12))

def gen():
    for n in range(12):
        yield n ** 2

G2 = gen()
print(*G1)
print(*G2)

Una función generadora es una función que, en lugar de usar ``return`` para devolver un valor una sola vez, usa ``yield`` para entregar una secuencia de valores (potencialmente infinita).
Igual que en las expresiones generadoras, el estado del generador se conserva entre recorridos parciales; si queremos una copia nueva del generador, basta con volver a llamar a la función.

> 📝 **Nota de MiAtlas.cl:** una función generadora puede delegar en otro iterable con ``yield from`` (disponible desde Python 3.3). Es lo mismo que recorrerlo con un ``for`` y hacer ``yield`` de cada elemento, pero más corto.

In [ ]:
def cuadrados_y_cubos(n):
    yield from (k ** 2 for k in range(n))
    yield from (k ** 3 for k in range(n))

print(*cuadrados_y_cubos(4))

## Ejemplo: generador de números primos
Aquí va mi ejemplo favorito de función generadora: una función que genera una serie ilimitada de números primos.
Un algoritmo clásico para esto es la *criba de Eratóstenes*, que funciona más o menos así:

In [ ]:
# Generar una lista de candidatos
L = [n for n in range(2, 40)]
print(L)

In [ ]:
# Quitar todos los múltiplos del primer valor
L = [n for n in L if n == L[0] or n % L[0] > 0]
print(L)

In [ ]:
# Quitar todos los múltiplos del segundo valor
L = [n for n in L if n == L[1] or n % L[1] > 0]
print(L)

In [ ]:
# Quitar todos los múltiplos del tercer valor
L = [n for n in L if n == L[2] or n % L[2] > 0]
print(L)

Si repetimos este procedimiento suficientes veces sobre una lista suficientemente grande, podemos generar todos los primos que queramos.

Encapsulemos esta lógica en una función generadora:

In [ ]:
def gen_primos(N):
    """Genera los primos menores que N"""
    primos = set()
    for n in range(2, N):
        if all(n % p > 0 for p in primos):
            primos.add(n)
            yield n

print(*gen_primos(100))

¡Eso es todo!
Aunque esta no es la implementación más eficiente de la criba de Eratóstenes, muestra lo cómoda que es la sintaxis de las funciones generadoras para construir secuencias más complicadas.

## Pruébalo tú

Crea una **expresión generadora** (no una lista) llamada `resultado` que produzca los **cubos de los números impares del 1 al 10**, es decir, 1³, 3³, 5³, 7³ y 9³.

*Pista:* usa paréntesis, ``range`` y una condición con ``%``.

In [ ]:
resultado = None  # escribe aquí tu expresión generadora

In [ ]:
# Comprobación
import types

if resultado is None:
    print("Aún no lo resuelves.")
elif isinstance(resultado, types.GeneratorType) and list(resultado) == [1, 27, 125, 343, 729]:
    print("✅ ¡Correcto!")
else:
    print("❌ Revisa tu respuesta")

**Sigue en MiAtlas.cl:** tema «Iteradores y generadores» · «Comprensiones y generadores».

**Original:** [12-Generators.ipynb](https://github.com/jakevdp/WhirlwindTourOfPython/blob/master/12-Generators.ipynb)